# Keithley

In [6]:
from pymeasure.instruments.keithley import Keithley2400

In [7]:
sourcemeter = Keithley2400("GPIB::4")

In [18]:
sourcemeter.voltage

0.3303812

In [20]:
sourcemeter.source_current

0.001

# Moku

## single instrument

In [1]:
from aparatus.sagnac1 import i


In [14]:
i.claim_ownership()

'194ddcbce9a'

In [9]:
i.relinquish_ownership()

In [2]:
i.sample()

True

In [6]:
i.sample.df.ch1.mean()

np.float64(-0.02518762026238619)

In [6]:
i.get_demodulation()

{'frequency': 3347620.0, 'phase': 160.0, 'source': 'ExternalPLL'}

In [8]:
i.set_gain(63,67)

{'aux': 67.0,
 'aux_gain_range': '0dB',
 'aux_invert': False,
 'main': 63.0,
 'main_gain_range': '0dB',
 'main_invert': False}

## multi instrument sideband demod

In [1]:
from aparatus.sagnac1 import SidebandDemod

m, wg, har2, har1, sideband = SidebandDemod()

[{'destination': 'Slot1InA', 'source': 'Slot4OutA'}, {'destination': 'Slot1InB', 'source': 'Slot4OutB'}, {'destination': 'Slot2InA', 'source': 'Input1'}, {'destination': 'Slot2InB', 'source': 'Slot1OutA'}, {'destination': 'Slot3InA', 'source': 'Input1'}, {'destination': 'Slot3InB', 'source': 'Slot1OutA'}, {'destination': 'Slot4InA', 'source': 'Input1'}, {'destination': 'Slot4InB', 'source': 'Slot1OutB'}, {'destination': 'Output1', 'source': 'Slot1OutA'}, {'destination': 'Output2', 'source': 'Slot1OutB'}]


In [2]:
m.sample()

LockInAmp (2036120571728): runtime -0.018999576568603516
LockInAmp (2036779863648): runtime -9.74129033088684
LockInAmp (2036443783456): runtime -0.02199864387512207


In [ ]:
har1

'Moku'

In [29]:
har2.set_trigger(mode='Auto')
har2.set_acquisition_mode(mode="Precision")

{'mode': 'Precision'}

In [8]:
import time
tic = time.time()
har2.get_data()
toc = time.time()
toc-tic

3.5313193798065186

In [ ]:
har1.start_streaming()

In [ ]:
har1.start_streaming(duration=0.1)
har1.aq

{'acquisition_mode': 'Normal', 'rate': 1000.0, 'stream_id': 'logsink2'}

In [ ]:
har1.start_streaming(duration=1)
for i in range(10):
    tic = time.time()
    har1.get_stream_data()
    toc = time.time()
    print(toc-tic)

In [24]:
# testing setting the Time constant
import numpy as np
m.setTc(0.0031)
round(1/ (har1.get_filter()["corner"] *2 * np.pi),5)

0.00311

In [6]:
m.claim_ownership()

'194e3b41345'

In [6]:
wg.relinquish_ownership()

In [12]:
m.relinquish_ownership()

## Hardcode

### single instrument

In [1]:
import time
from moku.instruments import LockInAmp
i = LockInAmp('[fe80::32e2:83ff:fea0:7141%2]', force_connect=True)
# i.sample()
i.relinquish_ownership()

### multi instrument

In [3]:
import time
from moku.instruments import MultiInstrument
from moku.instruments import WaveformGenerator, LockInAmp

m = MultiInstrument('[fe80::5871:e09a:a71e:c8cf%8]', platform_id=4,force_connect=True)
wg = m.set_instrument(1, WaveformGenerator)
har2 = m.set_instrument(2, LockInAmp)
har1 = m.set_instrument(3, LockInAmp)
sideband = m.set_instrument(4, LockInAmp)

connections = [ # Inputs
    dict(source="Input1", destination="Slot2InA"),
    dict(source="Input1", destination="Slot3InA"),
    dict(source="Slot3OutA", destination="Slot4InA"),
    # signal Generation PLL
    dict(source="Slot1OutA", destination="Slot2InB"),
    dict(source="Slot1OutA", destination="Slot3InB"),
    dict(source="Slot1OutB", destination="Slot4InB"),
    # Outputs
    dict(source="Slot1OutA", destination="Output1"),
    dict(source="Slot1OutB", destination="Output2")
    ]

print(m.set_connections(connections=connections))

m.set_frontend(1, coupling='AC', impedance='1MOhm', attenuation='-20dB')
# m.set_frontend(2, coupling='AC', impedance='1MOhm',attenuation='-20dB')






[{'destination': 'Slot1InA', 'source': 'Slot4OutA'}, {'destination': 'Slot1InB', 'source': 'Slot4OutB'}, {'destination': 'Slot2InA', 'source': 'Input1'}, {'destination': 'Slot2InB', 'source': 'Slot1OutA'}, {'destination': 'Slot3InA', 'source': 'Input1'}, {'destination': 'Slot3InB', 'source': 'Slot1OutA'}, {'destination': 'Slot4InA', 'source': 'Slot3OutA'}, {'destination': 'Slot4InB', 'source': 'Slot1OutB'}, {'destination': 'Output1', 'source': 'Slot1OutA'}, {'destination': 'Output2', 'source': 'Slot1OutB'}]


{'attenuation': '-20dB', 'coupling': 'AC', 'impedance': '1MOhm'}

In [ ]:
har1.set_acquisition_mode()

{'mode': 'Normal'}

In [23]:
har1.set_trigger()

{'auto_sensitivity': False,
 'edge': 'Rising',
 'hf_reject': False,
 'holdoff': 0.0,
 'hysteresis': 0.001,
 'level': 0.0,
 'noise_reject': False,
 'nth_event': 1}

In [22]:
har1.set_acquisition_mode("auto")

InvalidRequestException: ['Cannot understand request. One or more key/values are incorrect']

In [9]:
import pandas as pd
pd.DataFrame( har1.get_data() )

,ch1,ch2,time
0,0.0,0.0,-0.000500
1,0.0,0.0,-0.000499
2,0.0,0.0,-0.000498
3,0.0,0.0,-0.000497
4,0.0,0.0,-0.000496
...,...,...,...
1019,0.0,0.0,0.000508
1020,0.0,0.0,0.000509
1021,0.0,0.0,0.000510
1022,0.0,0.0,0.000511


In [19]:
( har1.get_data(measurements=True) )["measurements"]

{'ch1': {'amplitude': 0.0,
  'cycle_mean': None,
  'cycle_rms': None,
  'duty_cycle': None,
  'fall_rate': None,
  'fall_time': None,
  'frequency': None,
  'fringe_vis.': None,
  'high_level': 0.0,
  'low_level': 0.0,
  'maximum': 0.0,
  'mean': 0.0,
  'minimum': 0.0,
  'neg._width': None,
  'overshoot': 0.0,
  'peak_to_peak': 0.0,
  'period': None,
  'phase': None,
  'pulse_width': None,
  'rise_rate': None,
  'rise_time': None,
  'rms': 0.0,
  'std._deviation': 0.0,
  'undershoot': 0.0},
 'ch2': {'amplitude': 0.0,
  'cycle_mean': None,
  'cycle_rms': None,
  'duty_cycle': None,
  'fall_rate': None,
  'fall_time': None,
  'frequency': None,
  'fringe_vis.': None,
  'high_level': 0.0,
  'low_level': 0.0,
  'maximum': 0.0,
  'mean': 0.0,
  'minimum': 0.0,
  'neg._width': None,
  'overshoot': 0.0,
  'peak_to_peak': 0.0,
  'period': None,
  'phase': None,
  'pulse_width': None,
  'rise_rate': None,
  'rise_time': None,
  'rms': 0.0,
  'std._deviation': 0.0,
  'undershoot': 0.0}}

In [12]:
wg = WaveformGenerator('[fe80::32e2:83ff:fea0:7141%2]',force_connect=True)

In [15]:
wg.generate_waveform(channel=1, type="Sine",frequency=3e6, amplitude=0.5, offset=0, phase=0)

{'amplitude': 0.5,
 'frequency': 3000000.0,
 'offset': 0.0,
 'phase': 0.0,
 'type': 'Sine'}

In [2]:
m.relinquish_ownership()

NameError: name 'm' is not defined

# Junk

In [36]:
import numpy as np
print( np.arctan2(1.2,5), np.arctan(1.2/5) )

0.23554498072086333 0.23554498072086333


In [54]:
import numpy as np
Tc = 0.0102
1/(2*np.pi*Tc)

15.603425793323071

In [47]:
from types import MethodType
class junk:
    def __init__(self):
        self.i = 0

    def __iter__(self):
        return self

    def __next__(self):
        self.i += 1
        if self.i > 10:
            raise StopIteration
        return self.i
j = junk()

def sample(self):
    import numpy as np
    self.df = np.random.rand(100,2)
    return True
j.sample = MethodType(sample, j)

In [48]:
j.sample()

True

In [49]:
j.df

array([[8.18748826e-01, 4.58728155e-02],
       [3.26307645e-01, 7.11576486e-01],
       [1.39209111e-01, 7.97739645e-01],
       [7.83821539e-01, 4.26852835e-01],
       [9.59260314e-02, 1.41993007e-01],
       [3.57194928e-01, 4.90649742e-01],
       [7.11176687e-01, 3.39496970e-01],
       [2.89499329e-01, 4.54550839e-01],
       [6.57277519e-01, 9.42303995e-01],
       [7.86814209e-01, 4.79773356e-01],
       [5.58094069e-01, 2.24461928e-01],
       [6.83821363e-01, 2.88433052e-01],
       [6.46720793e-01, 1.05703128e-01],
       [9.70702583e-01, 6.99803756e-01],
       [3.62310506e-01, 5.71514846e-01],
       [2.12163017e-01, 4.91500606e-01],
       [2.87199547e-02, 9.75808918e-01],
       [5.57987101e-01, 2.11627460e-01],
       [4.59758527e-01, 5.82378253e-02],
       [4.39944762e-01, 9.29537233e-01],
       [3.64370037e-01, 9.65955990e-01],
       [2.26570992e-01, 3.21320580e-01],
       [5.69504620e-01, 5.04817651e-01],
       [6.33988244e-01, 3.59861196e-01],
       [6.083070

# Manual Control

In [1]:
from aparatus.sagnac1 import magnet, delayStage, myHF2LI


In [12]:
magnet.field =0

In [11]:
magnet.field = -0.17

print(f"mag field {magnet.Bx}, {magnet.By}, {magnet.Bz}")

mag field 0.16741731801207538, 0.02952019020337815, -0.0


In [6]:
myHF2LI.get("sigouts/0/amplitude")

KeyError: '/dev1004/get'

In [5]:
myHF2LI.dem(3)

{'timestamp': array([298551629496320], dtype=uint64),
 'x': array([0.00079251]),
 'y': array([-2.03789195e-05]),
 'frequency': array([3347620.00000008]),
 'phase': array([3.62744054]),
 'dio': array([1073741763], dtype=uint32),
 'trigger': array([3], dtype=uint32),
 'auxin0': array([-0.00823987]),
 'auxin1': array([-0.00244144])}

In [2]:
delayStage.y.position = 0

In [5]:
myHF2LI.setTcAll(1)

In [9]:
from experimenter import *
tc = 0.1
def setTc(Tc): [myHF2LI.demods[demod].timeconstant(Tc) for demod in range(6)]

In [1]:
from aparatus.sagnac1 import delayStage
def yoinkers(thing):
    thing = thing + 1
    return thing

delayStage.yoinkers = yoinkers

delayStage.yoinkers(1)

2

# dummy run

In [2]:
import experimenter
from aparatus.dummy import instrument as inst
import numpy as np
parameters = {
   "inst.phase": 0,
   "inst.frequency": np.linspace(1, 10, 5),
   "inst.temperature": [25, 30, 35]
}

namespace = {"np": np, "inst": inst, "parameters": parameters}

experimenter.perform_measurement("Tests/dummy.csv", namespace);

	Measuring...



In [3]:
experimenter.run_experiment(parameters, "Tests/dummy.csv",namespace, demoMode=True)

Running experiment with variables: {'inst.phase': 0, 'inst.frequency': array([ 1.  ,  3.25,  5.5 ,  7.75, 10.  ]), 'inst.temperature': [25, 30, 35]}
	Setting inst.phase to 0

Cartesian Product of remaining variables:
Running experiment with variables: {'inst.frequency': np.float64(1.0), 'inst.temperature': 25}
	Setting inst.frequency to 1.0
	Setting inst.temperature to 25
	Measuring...

Running experiment with variables: {'inst.frequency': np.float64(1.0), 'inst.temperature': 30}
	Setting inst.frequency to 1.0
	Setting inst.temperature to 30
	Measuring...

Running experiment with variables: {'inst.frequency': np.float64(1.0), 'inst.temperature': 35}
	Setting inst.frequency to 1.0
	Setting inst.temperature to 35
	Measuring...

Running experiment with variables: {'inst.frequency': np.float64(3.25), 'inst.temperature': 25}
	Setting inst.frequency to 3.25
	Setting inst.temperature to 25
	Measuring...

Running experiment with variables: {'inst.frequency': np.float64(3.25), 'inst.temperature

# delay stage stuff